# Geometric decision-making in a neural spin model

**DRAFT for review.** This notebook uses MuMoT to reproduce, in a simplified mean-field and stochastic form, the core mechanism of the neural *spin model* of spatial decision-making in

* V. H. Sridhar, L. Li, D. Gorbonos, M. Nagy, B. R. Schell, T. Sorochkin, N. S. Gov and I. D. Couzin (2021) *The geometry of decision-making in individuals and collectives*. PNAS **118**(50): e2102157118. [doi:10.1073/pnas.2102157118](https://doi.org/10.1073/pnas.2102157118)
* D. Gorbonos, N. S. Gov and I. D. Couzin (2024) *Geometrical structure of bifurcations during spatial decision-making*. PRX Life **2**: 013008. [doi:10.1103/PRXLife.2.013008](https://doi.org/10.1103/PRXLife.2.013008)

In these papers an animal moving towards several targets represents each target direction with a group of neurons, modelled as Ising-like spins (as in a Hopfield network). Neurons coding for the same direction excite each other; the interaction between groups depends on the angle between the target directions. When the targets are close together in angle the animal *compromises*, moving in the average direction; beyond a critical angle it abruptly *decides*, moving towards one target. With more targets this happens repeatedly, breaking a multi-choice decision into a series of binary ones.

MuMoT cannot simulate the embodied agent (its movement changes the angles to the targets over time, which drives the sequence of bifurcations in the papers). What it can do is analyse the neural decision dynamics at **fixed** target angles: steady states, their stability, the bifurcation from compromise to decision, and the effect of finite numbers of neurons (noise).

> **Notes for the reviewer** are marked like this. In particular the spin-model details below were written without access to the papers' full text and must be checked against them.

In [ ]:
import mumot
import numpy as np

## From the spin model to a MuMoT model

In the spin model each neuron $i$ is a spin $\sigma_i \in \{0, 1\}$ (inactive/active), assigned to one target direction. The energy is of Hopfield form

$$H = -\frac{k}{N} \sum_{i \neq j} J_{ij}\, \sigma_i \sigma_j, \qquad J_{ij} = \cos\left(\pi \left(\frac{|\theta_{ij}|}{\pi}\right)^{\nu}\right),$$

where $\theta_{ij}$ is the angle between the target directions coded by neurons $i$ and $j$ (so $J_{ij} = 1$ within a group), and $\nu$ controls how the brain represents angles ($\nu = 1$: the plain cosine). Spins flip stochastically at a 'temperature' (neural noise), and the agent moves in the average direction of the targets coded by active neurons.

> **Reviewer:** check the exact Hamiltonian, normalisation ($k$, $N$, any constraint on total activity), the dynamics (Glauber/Metropolis) and the definition of $\nu$ against the papers.

To express this in MuMoT we treat the neurons as a population with three states: active for target 1 ($A$), active for target 2 ($B$) and inactive ($U$), and replace the spin-flip dynamics by mass-action reactions with the same sign structure as the couplings:

| spin model | reaction | rate |
|---|---|---|
| spontaneous activation (noise) | $U \to A$, $U \to B$ | $g$ |
| spontaneous deactivation | $A \to U$, $B \to U$ | $a$ |
| excitation within a group ($J = 1$) | $A + U \to A + A$, $B + U \to B + B$ | $r$ |
| excitation between groups ($J_{AB} > 0$, acute angles) | $B + U \to B + A$, $A + U \to A + B$ | $c_E = r \max(J_{AB}, 0)$ |
| inhibition between groups ($J_{AB} < 0$, obtuse angles) | $A + B \to A + U$, $A + B \to B + U$ | $c_I = r \max(-J_{AB}, 0)$ |

With $c_E = 0$ this is exactly the cross-inhibition ('stop-signal') model of house-hunting honeybees analysed earlier in the user manual, here with equal option values: the neural spin model and the collective decision model share the same decision mechanism.

In [ ]:
%%model
$
U -> A : g
U -> B : g
A -> U : a
B -> U : a
A + U -> A + A : r
B + U -> B + B : r
B + U -> B + A : c_E
A + U -> A + B : c_E
A + B -> A + U : c_I
A + B -> B + U : c_I
$

In [ ]:
model1 = mumot.parseModel(In[-2])

In [ ]:
model1.visualise()

The total number of neurons is fixed, so we eliminate the inactive state, $U = N - A - B$, and look at the mean-field equations for the fractions of active neurons $\Phi_A$ and $\Phi_B$:

In [ ]:
model2 = model1.substitute('U = N - A - B')
model2.showODEs()

### Couplings from the geometry

A small helper converts the angle between the two targets into the coupling rates, for a given angle representation $\nu$ and within-group excitation $r$.

In [ ]:
def coupling(theta_deg, nu=1.0, r=4.0):
    """Between-group coupling J and the rates (c_E, c_I) for targets theta_deg degrees apart."""
    theta = np.radians(theta_deg)
    J = np.cos(np.pi * (theta / np.pi)**nu)
    return J, r * max(J, 0.0), r * max(-J, 0.0)

for theta in (30, 60, 90, 120, 150):
    J, c_E, c_I = coupling(theta)
    print(f"theta = {theta:3d} deg: J = {J:+.2f}, c_E = {c_E:.2f}, c_I = {c_I:.2f}")

## Compromise and decision

We use $g = 0.1$ (noise), $a = 1$ and $r = 4$ throughout (relative to $a$). First, two targets $60°$ apart: the groups excite each other, both stay equally active and the agent heads between the targets (compromise).

In [ ]:
J, c_E, c_I = coupling(60)
compromise = model2.integrate(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                              initWidgets={'initialState': {'A': [0.12, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                            'U': [0.78, 0, 1, 0.01]},
                                           'maxTime': [10, 1, 50, 1]}, plotProportions=True)

Now $120°$ apart: the groups inhibit each other and a slight initial advantage for target 1 grows until one group dominates (decision).

In [ ]:
J, c_E, c_I = coupling(120)
decision = model2.integrate(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                            initWidgets={'initialState': {'A': [0.12, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                          'U': [0.78, 0, 1, 0.01]},
                                         'maxTime': [10, 1, 50, 1]}, plotProportions=True)

The phase planes show why. A multicontroller puts both views in one figure, with a single set of sliders for the parameters they share (noise $g$, decay $a$ and excitation $r$), while each view keeps its own couplings: at $60°$ (left) there is a single stable state on the diagonal $\Phi_A = \Phi_B$; at $120°$ (right) that state has become unstable (a saddle), flanked by two stable decision states.

In [ ]:
J, c_E60, c_I60 = coupling(60)
J, c_E120, c_I120 = coupling(120)
phasePlanes = mumot.MuMoTmultiController(
    [model2.stream('A', 'B', showFixedPoints=True, params=[('c_E', c_E60), ('c_I', c_I60)], silent=True),
     model2.stream('A', 'B', showFixedPoints=True, params=[('c_E', c_E120), ('c_I', c_I120)], silent=True)],
    initWidgets={'g': [0.1, 0, 1, 0.05], 'a': [1, 0.1, 2, 0.1], 'r': [4, 0, 10, 0.5]})

## The bifurcation

For obtuse angles ($c_E = 0$) the inhibition $c_I = -rJ$ grows with the angle. Continuing the steady states in $c_I$ shows a pitchfork bifurcation: the compromise state loses stability at a critical inhibition (BP1) and two symmetric decision branches appear. Combining bifurcation views for three noise levels in one figure shows how noise delays the decision: from left to right the branch points are for $g = 0.05$, $0.2$ and $0.4$ (the other parameters are shared, on the sliders).

In [ ]:
noiseLevels = mumot.MuMoTmultiController(
    [model2.bifurcation('c_I', 'A-B', params=[('c_E', 0), ('g', g)], silent=True,
                        initWidgets={'initBifParam': [0.05, 0.01, 4, 0.01]})
     for g in (0.05, 0.2, 0.4)],
    shareAxes=True, initWidgets={'a': [1, 0.1, 2, 0.1], 'r': [4, 0, 10, 0.5]},
    choose_xrange=[0, 4], xlab=r'inhibition $c_I$')

### From critical coupling to critical angle

For this model the critical coupling can also be found by hand. In the compromise state $\Phi_A = \Phi_B$ the difference $\Phi_A - \Phi_B$ grows at rate $r\Phi_U - a$ (the cross-inhibition terms cancel), so the compromise becomes unstable when the fraction of inactive neurons falls to $\Phi_U = a/r$. Putting this into the steady-state equations gives

$$c_{I,c} = \frac{4\,g\,a\,r}{(r - a)^2},$$

which matches the branch points above ($0.089$, $0.356$ and $0.711$ for $g = 0.05$, $0.2$ and $0.4$, with $a = 1$, $r = 4$). Inverting $c_I = -r\cos\left(\pi(\theta/\pi)^\nu\right)$ gives the critical angle between the targets,

$$\theta_c = \pi\left(\frac{\arccos(-c_{I,c}/r)}{\pi}\right)^{1/\nu}.$$

Below $\theta_c$ the agent compromises, heading between the targets; above it, it heads for one of them. Noise increases the critical angle, while $\nu < 1$ (a compressed representation of angles) makes the decision happen at much smaller angles.

> **Reviewer:** compare with the critical angles in the papers (for the values of $\nu$ they use, and how they depend on noise). The notebook's $g$ plays the role of neural noise and $r$ of the coupling strength $k$ (inverse temperature).

In [ ]:
def critical_angle(g, a=1.0, r=4.0, nu=1.0):
    """Angle between the targets (degrees) at which the compromise becomes unstable."""
    c_I_crit = 4 * g * a * r / (r - a)**2
    return np.degrees(np.pi * (np.arccos(-c_I_crit / r) / np.pi)**(1 / nu))

for g in (0.05, 0.1, 0.2, 0.4):
    print(f"noise g = {g:4}: critical angle {critical_angle(g):5.1f} deg (nu = 1), "
          f"{critical_angle(g, nu=0.5):5.1f} deg (nu = 0.5)")

## Finite numbers of neurons

With a finite number of neurons $N$ the decision is stochastic. Above the critical angle, repeated simulations decide for either target at random; near it, fluctuations make the decision slow and noisy. Use the *System size* slider to see the effect of $N$.

In [ ]:
J, c_E, c_I = coupling(120)
ssa1 = model2.SSA(params=[('g', 0.1), ('a', 1), ('r', 4), ('c_E', c_E), ('c_I', c_I)],
                  initWidgets={'initialState': {'A': [0, 0, 1, 0.01], 'B': [0, 0, 1, 0.01], 'U': [1, 0, 1, 0.01]},
                               'maxTime': [20, 1, 100, 1], 'systemSize': [50, 10, 500, 10], 'runs': [5, 1, 20, 1],
                               'aggregateResults': False}, plotProportions=True)

## Three targets (exploratory)

> **Reviewer:** this section is the least certain. In the papers the sequence of bifurcations with three or more targets arises as the moving agent's angles to the targets change. Here the angles are fixed, so only snapshots of that process can be studied.

With three targets at $-\theta$, $0$ and $+\theta$ (neurons $A$, $B$, $C$), each pair interacts according to its own angle: pairs 1 ($A$-$B$) and 2 ($B$-$C$) are adjacent, $\theta$ apart, and pair 3 ($A$-$C$) is $2\theta$ apart. (The rates are numbered because MuMoT would read a subscript such as $e_{AB}$ as $e_{A \cdot B}$.)

In [ ]:
%%model
$
U -> A : g
U -> B : g
U -> C : g
A -> U : a
B -> U : a
C -> U : a
A + U -> A + A : r
B + U -> B + B : r
C + U -> C + C : r
B + U -> B + A : e_1
A + U -> A + B : e_1
C + U -> C + B : e_2
B + U -> B + C : e_2
C + U -> C + A : e_3
A + U -> A + C : e_3
A + B -> A + U : i_1
A + B -> B + U : i_1
B + C -> B + U : i_2
B + C -> C + U : i_2
A + C -> A + U : i_3
A + C -> C + U : i_3
$

In [ ]:
model3 = mumot.parseModel(In[-2]).substitute('U = N - A - B - C')

In [ ]:
def three_target_params(theta_deg, nu=1.0, r=4.0):
    """Partial-controller parameters for targets at -theta, 0, +theta."""
    # pairs 1: A-B, 2: B-C (adjacent targets), 3: A-C (outer targets)
    angles = {'1': theta_deg, '2': theta_deg, '3': min(2 * theta_deg, 360 - 2 * theta_deg)}
    p = [('g', 0.1), ('a', 1), ('r', r)]
    for pair, angle in angles.items():
        J, c_E, c_I = coupling(angle, nu, r)
        p += [('e_' + pair, c_E), ('i_' + pair, c_I)]
    return p

three60 = model3.integrate(params=three_target_params(60),
                           initWidgets={'initialState': {'A': [0.11, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                         'C': [0.1, 0, 1, 0.01], 'U': [0.69, 0, 1, 0.01]},
                                        'maxTime': [20, 1, 50, 1]}, plotProportions=True)

In [ ]:
three130 = model3.integrate(params=three_target_params(130),
                            initWidgets={'initialState': {'A': [0.11, 0, 1, 0.01], 'B': [0.1, 0, 1, 0.01],
                                                          'C': [0.1, 0, 1, 0.01], 'U': [0.69, 0, 1, 0.01]},
                                         'maxTime': [20, 1, 50, 1]}, plotProportions=True)

At $60°$ all three groups stay active: a compromise heading towards the middle target. At $130°$ the decision unfolds as a sequence of binary decisions in time, as in the papers: first the middle target is excluded (the outer groups both inhibit it), then the two outer groups compete until one wins. Use the sliders to explore other angles and noise levels.

## Open questions for review

1. Check the spin-model description (Hamiltonian, $J_{ij}$, $\nu$, noise, dynamics) and the citations against the papers.
2. Is the split of $J$ into excitation ($J > 0$) and inhibition ($J < 0$) reactions the right mass-action analogue of the spin dynamics? An alternative is $c_E = r(1 + J)/2$, $c_I = r(1 - J)/2$, which keeps the net coupling $rJ$ but adds both terms at every angle (it gives a critical angle of about $115°$ for $\nu = 1$ instead of about $95°$, and different three-target behaviour).
3. Which quantitative results from the papers (critical angles for given $\nu$ and noise) should the notebook aim to match, and with which parameter values?
4. Should the three-target section reproduce a specific figure, e.g. by choosing angles that correspond to points along an agent's trajectory?